# Contrail-detector filter playground

Interactive explorer for the detector redesign (PRD item 29). Stack up to
three preprocessing transforms, tune Canny + Hough, and press **Bulk AUC**
to see how the chain scores against the April-9 35-label batch plus the 7
April-8 positive sentinels.

## What this notebook shows

Four panels per candidate:
1. **Crop + rotated polygon** — the frame the pipeline hands to `detect()`.
2. **Chain output (gray)** — what `apply_chain` produces, using the current
   transform chain and per-transform parameter sliders.
3. **Canny edges** — edges inside the rotated polygon mask, using the
   post-chain gray image.
4. **Hough overlay** — aligned long lines in green, rejected lines in
   orange, picked `pixel_line` in thick bright-green.

## GO/NO-GO (from PRD item 29)

On the April-9 label set (15 positives / 19 negatives), the winning chain
and Youden-J threshold must achieve:
- recall ≥ 10/15 (≥66%)
- false-positive count ≤ 2/19 (≤10%)
- non-zero recall on at least 3 of 4 strata (high_cirrus, mid_cruise, wide_radius, marginal)

Baseline: the current production detector catches only **1/15** April-9
positives. Anything meaningfully above that is progress.

## Setup

```
uv sync --extra review
uv run jupyter lab notebooks/filter_playground.ipynb
```

In [10]:
from __future__ import annotations

import datetime
import json
import math
import sys
from collections import defaultdict
from datetime import timezone
from pathlib import Path

import av
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import (
    Button, Checkbox, Dropdown, FloatSlider, HBox, IntSlider, Label, Layout,
    Output, VBox, interactive_output,
)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from concam.adsb import Ping
from concam.config import DetectionConfig, load_config
from concam.detection import detect
from concam.detection.transforms import (
    DISPLAY_COLORMAPS, DISPLAY_LABELS, TRANSFORMS, TRANSFORMS_BY_NAME, apply_chain,
)
from concam.projection import (
    Calibration, PixelPoint, Rect, load_calibration, project_pings, rotated_polygon,
)

SITE_CONFIG = load_config(REPO_ROOT / "configs" / "mit_green_building.yaml")
DET_CFG = SITE_CONFIG.detection
CALIB = load_calibration(SITE_CONFIG.calibration)
CALIB_W, CALIB_H = (
    int(SITE_CONFIG.calibration.calibration_resolution[0]),
    int(SITE_CONFIG.calibration.calibration_resolution[1]),
)
EXTRACT_PAD = 20

APRIL9_DIR = REPO_ROOT / "output" / "validation" / "detection" / "2026-04-09" / "label_batch"
APRIL9_CANDIDATES_PATH = APRIL9_DIR / "candidates.json"
APRIL9_LABELS_PATH = REPO_ROOT / "labels" / "2026-04-09_prash.json"
APRIL9_VIDEO_LINK = APRIL9_DIR / "video.mp4"
APRIL9_ADSB_PATH = REPO_ROOT / "output" / "2026-04-09" / "adsb.json"
APRIL9_OCR_PATH = REPO_ROOT / "output" / "2026-04-09" / "ocr.jsonl"

APRIL8_DIR = REPO_ROOT / "output" / "validation" / "detection" / "2026-04-08"
APRIL8_MANIFEST_PATH = APRIL8_DIR / "manifest.json"
APRIL8_LABELS_PATH = APRIL8_DIR / "labels.json"

# Load April-9 ADS-B once so both the batch loader and the user-candidate
# ingester can look up ping sequences for runtime path_vec recomputation.
ADSB = json.loads(APRIL9_ADSB_PATH.read_text())
ADSB_BY_TID: dict[str, dict] = {f["transponder_id"]: f for f in ADSB}

# Video t0 for wall_time <-> frame_idx conversion (1 fps timelapse).
with open(APRIL9_OCR_PATH) as _f:
    _first = json.loads(_f.readline())
VIDEO_T0 = datetime.datetime.fromisoformat(_first["wall_time_utc"])

print(f"April-9 candidates: {APRIL9_CANDIDATES_PATH}")
print(f"April-9 labels:     {APRIL9_LABELS_PATH}")
print(f"April-9 video link: {APRIL9_VIDEO_LINK}")
print(f"April-9 adsb:       {APRIL9_ADSB_PATH} ({len(ADSB)} flights)")
print(f"April-8 manifest:   {APRIL8_MANIFEST_PATH}")

April-9 candidates: /home/prash/contrails/mit-concam-pipeline/output/validation/detection/2026-04-09/label_batch/candidates.json
April-9 labels:     /home/prash/contrails/mit-concam-pipeline/labels/2026-04-09_prash.json
April-9 video link: /home/prash/contrails/mit-concam-pipeline/output/validation/detection/2026-04-09/label_batch/video.mp4
April-8 manifest:   /home/prash/contrails/mit-concam-pipeline/output/validation/detection/2026-04-08/manifest.json


## Load candidates

**April-9** candidates (34 labeled, episode 29 skipped) come from
`candidates.json` + `labels/2026-04-09_prash.json`. The peak frame for each
candidate is decoded from the full-day video; the frame at `peak_frame_idx - 1`
is also decoded so the `temporal_diff` transform has a prev-frame to subtract.

**April-8** positive sentinels come from the existing batch-1 `manifest.json`
+ `labels.json`. Crops are read from the pre-rendered `rois/roi_NN.png` files
on disk. These 7 positives are a regression safety-net: a chain that beats
April-9 but tanks on April-8 probably overfits to the new set.

In [11]:
def _context_crop(frame, pixel_x, pixel_y, pad_px):
    h, w = frame.shape[:2]
    cx, cy = int(round(pixel_x)), int(round(pixel_y))
    x1 = max(0, cx - pad_px)
    y1 = max(0, cy - pad_px)
    x2 = min(w, cx + pad_px)
    y2 = min(h, cy + pad_px)
    return frame[y1:y2, x1:x2].copy(), (x1, y1)


CROP_PAD_PX = 400  # half-width of the 800x800 detect+display crop


def _decode_frames(video_path: Path, target_indices: set[int]) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    container = av.open(str(video_path))
    try:
        stream = container.streams.video[0]
        time_base = float(stream.time_base)
        duration_s = float(stream.duration * stream.time_base) if stream.duration else 0.0
        total = int(stream.frames) if stream.frames else max(1, int(
            round(duration_s * float(stream.average_rate or 30))
        ))
        for target in sorted(target_indices):
            target_time = (target / total) * duration_s
            target_pts = int(target_time / time_base)
            container.seek(target_pts, stream=stream, any_frame=False, backward=True)
            decoded = None
            for frame in container.decode(stream):
                decoded = frame
                if frame.pts is not None and frame.pts >= target_pts:
                    break
            if decoded is not None:
                out[target] = decoded.to_ndarray(format="bgr24")
    finally:
        container.close()
    return out


def _find_ping_index(pings: list[dict], target_iso: str) -> int | None:
    """Return the index of the ping closest to target_iso in the upsampled
    1-second ping sequence. Returns None if no ping is within 60 s."""
    t_target = datetime.datetime.fromisoformat(target_iso).astimezone(timezone.utc)
    best_i, best_dt = None, None
    for i, p in enumerate(pings):
        pt = datetime.datetime.fromisoformat(p["time"]).astimezone(timezone.utc)
        dt = abs((pt - t_target).total_seconds())
        if best_dt is None or dt < best_dt:
            best_dt = dt
            best_i = i
    if best_dt is None or best_dt > 60.0:
        return None
    return best_i


april9_cands_raw = json.loads(APRIL9_CANDIDATES_PATH.read_text())["candidates"]
april9_labels_raw = json.loads(APRIL9_LABELS_PATH.read_text())["labels"]
label_by_eid = {l["episode_id"]: l for l in april9_labels_raw}

needed_frames: set[int] = set()
for c in april9_cands_raw:
    fidx = int(c["peak_frame_idx"])
    needed_frames.add(fidx)
    if fidx > 0:
        needed_frames.add(fidx - 1)

print(f"April-9: decoding {len(needed_frames)} frames from {APRIL9_VIDEO_LINK}")
if not APRIL9_VIDEO_LINK.exists():
    raise FileNotFoundError(
        f"video.mp4 symlink missing at {APRIL9_VIDEO_LINK}. "
        "Regenerate the label batch with scripts/build_april9_label_batch.py"
    )
frames = _decode_frames(APRIL9_VIDEO_LINK, needed_frames)
print(f"  decoded {len(frames)} frames")

april9_records: list[dict] = []
for c in april9_cands_raw:
    fidx = int(c["peak_frame_idx"])
    frame = frames.get(fidx)
    if frame is None:
        print(f"  SKIP episode {c['episode_id']}: frame {fidx} not decoded")
        continue
    crop, (tl_x, tl_y) = _context_crop(
        frame, float(c["pixel_x"]), float(c["pixel_y"]), CROP_PAD_PX
    )
    prev_crop = None
    if fidx > 0 and (fidx - 1) in frames:
        prev_crop, _ = _context_crop(
            frames[fidx - 1], float(c["pixel_x"]), float(c["pixel_y"]), CROP_PAD_PX
        )
    label_raw = label_by_eid.get(c["episode_id"])
    label = None
    if label_raw is not None:
        if label_raw["label"] == "contrail":
            label = "positive"
        elif label_raw["label"] == "no_contrail":
            label = "negative"
        else:
            label = "unsure"
    # Attach ping sequence + peak index so path_vec can be recomputed on-the-fly
    # with any baseline the user picks via the slider.
    tid = c["transponder_id"]
    flight = ADSB_BY_TID.get(tid)
    pings: list[dict] = []
    peak_ping_idx: int | None = None
    if flight is not None:
        pings = flight["pings"]
        peak_ping_idx = _find_ping_index(pings, c["peak_wall_time"])
    april9_records.append({
        "source": "april-9",
        "id": f"A9#{c['episode_id']:02d}",
        "label": label,
        "stratum": c.get("stratum", "unknown"),
        "callsign": c["callsign"],
        "transponder_id": tid,
        "frame_idx": fidx,
        "pixel_x": float(c["pixel_x"]),
        "pixel_y": float(c["pixel_y"]),
        "path_dx": float(c["path_dx"]),  # pipeline-computed (1s baseline), used as fallback
        "path_dy": float(c["path_dy"]),
        "roi": c["roi"],
        "crop": crop,
        "prev_crop": prev_crop,
        "crop_tl": (tl_x, tl_y),
        "pings": pings,
        "peak_ping_idx": peak_ping_idx,
    })

pos = sum(1 for r in april9_records if r["label"] == "positive")
neg = sum(1 for r in april9_records if r["label"] == "negative")
unsure = sum(1 for r in april9_records if r["label"] == "unsure")
unlabeled = sum(1 for r in april9_records if r["label"] is None)
with_pings = sum(1 for r in april9_records if r["peak_ping_idx"] is not None)
print(
    f"April-9 loaded: {len(april9_records)} records "
    f"(pos={pos}, neg={neg}, unsure={unsure}, unlabeled={unlabeled}, "
    f"with_pings={with_pings})"
)

April-9: decoding 70 frames from /home/prash/contrails/mit-concam-pipeline/output/validation/detection/2026-04-09/label_batch/video.mp4
  decoded 70 frames
April-9 loaded: 35 records (pos=15, neg=19, unsure=0, unlabeled=1)


In [12]:
april8_records: list[dict] = []
if APRIL8_MANIFEST_PATH.exists() and APRIL8_LABELS_PATH.exists():
    april8_manifest = json.loads(APRIL8_MANIFEST_PATH.read_text())
    april8_labels = json.loads(APRIL8_LABELS_PATH.read_text())
    label_by_idx = {e["idx"]: e["label"] for e in april8_labels["labels"]}
    for c in april8_manifest["candidates"]:
        label = label_by_idx.get(c["idx"])
        if label != "positive":
            continue
        crop = None
        crop_tl: tuple[int, int] | None = None
        ctx_png_rel = c.get("context_png")
        if ctx_png_rel:
            ctx_path = APRIL8_DIR / ctx_png_rel
            if ctx_path.exists():
                ctx_img = cv2.imread(str(ctx_path))
                if ctx_img is not None:
                    crop = ctx_img
                    ctx_h, ctx_w = ctx_img.shape[:2]
                    crop_tl = (
                        max(0, int(c["pixel_x"]) - ctx_w // 2),
                        max(0, int(c["pixel_y"]) - ctx_h // 2),
                    )
        if crop is None:
            roi_png_path = APRIL8_DIR / c["roi_png"]
            crop = cv2.imread(str(roi_png_path))
            if crop is None:
                print(f"  SKIP April-8 idx {c['idx']}: failed to read {roi_png_path}")
                continue
            crop_tl = (
                max(0, int(c["roi"]["x"]) - EXTRACT_PAD),
                max(0, int(c["roi"]["y"]) - EXTRACT_PAD),
            )
        april8_records.append({
            "source": "april-8",
            "id": f"A8#{c['idx']:02d}",
            "label": "positive",
            "stratum": "sentinel",
            "callsign": c["callsign"],
            "transponder_id": c.get("transponder_id", ""),
            "frame_idx": int(c["frame_idx"]),
            "pixel_x": float(c["pixel_x"]),
            "pixel_y": float(c["pixel_y"]),
            "path_dx": float(c["path_dx"]),
            "path_dy": float(c["path_dy"]),
            "roi": c["roi"],
            "crop": crop,
            "prev_crop": None,
            "crop_tl": crop_tl,
            # April-8 sentinels have no ping sequence here (different date's
            # adsb.json wasn't loaded). Fall back to stored path_dx/dy.
            "pings": [],
            "peak_ping_idx": None,
        })
    print(f"April-8 sentinels loaded: {len(april8_records)} positives")
else:
    print("April-8 sentinels skipped (manifest or labels missing)")

RECORDS: list[dict] = april9_records + april8_records
print(f"Total candidate records: {len(RECORDS)}")

April-8 sentinels loaded: 7 positives
Total candidate records: 42


In [13]:
def _ping_from_dict(d: dict) -> Ping:
    return Ping(
        time=datetime.datetime.fromisoformat(d["time"]),
        lat=float(d["lat"]),
        lon=float(d["lon"]),
        alt_m=float(d["alt_m"]),
        alt_gnss_m=d.get("alt_gnss_m"),
        alt_baro_m=d.get("alt_baro_m"),
        alt_source=d.get("alt_source", "gnss"),
    )


def get_path_vec(rec: dict, baseline_s: int) -> tuple[float, float]:
    """Compute path_vec (unit pixel-space vector) using a symmetric baseline.

    Uses pings at peak_ping_idx ± baseline_s (falling back to the stored
    path_dx/dy if the record has no pings, e.g. April-8 sentinels).  The
    production pipeline uses a 1-second baseline (adjacent upsampled pings),
    which is noisy at low displacements.  Larger baselines give more stable
    direction estimates at the cost of slight curvature aggregation over turns.
    """
    pings = rec.get("pings") or []
    i_peak = rec.get("peak_ping_idx")
    if not pings or i_peak is None or baseline_s < 1:
        return (float(rec["path_dx"]), float(rec["path_dy"]))
    i_back = max(0, i_peak - int(baseline_s))
    i_fwd = min(len(pings) - 1, i_peak + int(baseline_s))
    if i_fwd == i_back:
        return (float(rec["path_dx"]), float(rec["path_dy"]))
    back_ping = _ping_from_dict(pings[i_back])
    fwd_ping = _ping_from_dict(pings[i_fwd])
    projected = project_pings([back_ping, fwd_ping], CALIB)
    if projected[0] is None or projected[1] is None:
        return (float(rec["path_dx"]), float(rec["path_dy"]))
    dx = projected[1].x - projected[0].x
    dy = projected[1].y - projected[0].y
    mag = math.hypot(dx, dy)
    if mag < 1e-3:
        return (float(rec["path_dx"]), float(rec["path_dy"]))
    return (dx / mag, dy / mag)


def reconstruct_geometry(
    rec: dict, roi_along_px: int, roi_cross_px: int, path_baseline_s: int,
) -> tuple[Rect, np.ndarray, tuple[float, float]]:
    """Return (rect, polygon, path_vec) for a candidate record."""
    ch, cw = rec["crop"].shape[:2]
    tl_x, tl_y = rec["crop_tl"]
    center = PixelPoint(
        x=rec["pixel_x"] - tl_x,
        y=rec["pixel_y"] - tl_y,
    )
    path_vec = get_path_vec(rec, int(path_baseline_s))
    dummy = DetectionConfig(
        roi_along_px=int(roi_along_px),
        roi_cross_px=int(roi_cross_px),
        roi_padding=20,
    )
    poly = rotated_polygon(center, path_vec, dummy)
    rect = Rect(x=0, y=0, w=cw, h=ch)
    return rect, poly, path_vec


def build_config(knobs: dict) -> DetectionConfig:
    return DetectionConfig(
        score_threshold=0.3,
        canny_low=int(knobs["canny_low"]),
        canny_high=int(knobs["canny_high"]),
        hough_threshold=int(knobs["hough_threshold"]),
        hough_min_line_length=int(knobs["hough_min_line_length"]),
        hough_max_line_gap=int(knobs["hough_max_line_gap"]),
        roi_padding=20,
        roi_along_px=int(knobs["roi_along_px"]),
        roi_cross_px=int(knobs["roi_cross_px"]),
        use_adaptive_canny=bool(knobs["use_adaptive_canny"]),
        canny_percentile_low=float(knobs["canny_percentile_low"]),
        canny_percentile_high=float(knobs["canny_percentile_high"]),
        canny_low_ratio=float(knobs["canny_low_ratio"]),
        canny_min_high=int(knobs["canny_min_high"]),
        angle_tolerance_deg=float(knobs["angle_tolerance_deg"]),
        long_line_min_px=float(knobs["long_line_min_px"]),
        score_fn="length",
        score_length_norm_px=float(knobs["score_length_norm_px"]),
        score_norm_count=6,
        use_rotated_mask=bool(knobs["use_rotated_mask"]),
        blur_kernel=int(knobs["blur_kernel"]),
        preprocessing="none",
    )


def run_chain_and_detect(rec: dict, chain: list[str], transform_params: dict, knobs: dict):
    rect, poly, path_vec = reconstruct_geometry(
        rec,
        int(knobs["roi_along_px"]),
        int(knobs["roi_cross_px"]),
        int(knobs["path_baseline_s"]),
    )
    gray = apply_chain(
        rec["crop"], chain,
        path_vec=path_vec,
        prev_bgr=rec["prev_crop"],
        transform_params=transform_params,
    )
    cfg = build_config(knobs)
    result = detect(gray, rect, cfg, polygon=poly, path_vec=path_vec)
    return gray, result, rect, poly, path_vec, cfg

## Knob glossary

The sliders below drive two things: **what preprocessing is applied before
Canny** (the transform chain) and **how Canny + Hough behave on the result**.
Here's what the trickiest six knobs do — carry-over from the 2026-04-16
grill-me session.

- **prev_frame** (implicit, via the `temporal_diff` transform): the absolute
  difference between this frame and the one 1 s earlier. Static clouds
  cancel; newly-formed contrails remain. At 1 fps timelapse rates the diff
  is noisier than at 4 fps raw segments, so `temporal_diff` works best with
  low-gain settings or stacked after a spatial smoother.
- **pct_high / canny_percentile_high**: the Canny high threshold is set to
  this percentile of the masked-pixel distribution. Bigger → only the
  brightest edges survive. 99.0 catches fainter contrails; 99.8 is strict.
- **pct_low / canny_percentile_low**: the pixel-floor percentile. Everything
  dimmer than this percentile is zeroed out *before* Canny so low-contrast
  sky texture can't trigger edges. Typical 96–98.
- **low_ratio / canny_low_ratio**: `canny_low = canny_low_ratio * canny_high`.
  Sets the hysteresis width. 0.25 is a wide hysteresis (picks up weak edges
  that connect to strong ones); 0.5–0.8 is a narrow hysteresis (edge must
  be nearly as strong as the strongest).
- **min_high / canny_min_high**: floor on `canny_high`. Prevents a
  low-contrast crop from collapsing both thresholds to near-zero and lighting
  up on pixel noise. Default 60.
- **score_length_norm_px**: the along-track pixel length at which the
  detector score saturates at 1.0. Current production = 130 px, so a
  contrail spanning ≥130 px of the ROI scores 1.0. Lowering it makes short
  contrails score higher; raising it differentiates long contrails from very
  long ones. Replaced the discrete `score_norm_count=6` gate in PRD item 26.

In [14]:
# --- Candidate dropdown ---
def _label_tag(r):
    return r["label"] or "unlabeled"

_full = Layout(width="92%")
_half = Layout(width="46%")

cand_choices = [
    (f"{r['id']}  {r['callsign']:<10}  [{_label_tag(r)}, {r['stratum']}]", i)
    for i, r in enumerate(RECORDS)
]
w_cand = Dropdown(options=cand_choices, description="candidate", layout=_full)

# --- Transform-chain dropdowns (A -> B -> C) ---
chain_options = ["none"] + [t[0] for t in TRANSFORMS]
w_ch1 = Dropdown(options=chain_options, value="none", description="chain A", layout=_half)
w_ch2 = Dropdown(options=chain_options, value="none", description="chain B", layout=_half)
w_ch3 = Dropdown(options=chain_options, value="none", description="chain C", layout=_half)

# --- Per-transform parameter sliders ---
w_lc_sigma = FloatSlider(value=25.0, min=5.0, max=60.0, step=1.0, description="lc_sigma", layout=_half)
w_dog_lo = FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description="dog_lo", layout=_half)
w_dog_hi = FloatSlider(value=15.0, min=5.0, max=40.0, step=1.0, description="dog_hi", layout=_half)
w_frangi_lo = FloatSlider(value=1.0, min=0.5, max=5.0, step=0.5, description="frangi_σ_lo", layout=_half)
w_frangi_hi = FloatSlider(value=4.0, min=1.0, max=10.0, step=0.5, description="frangi_σ_hi", layout=_half)
w_clahe_clip = FloatSlider(value=3.0, min=1.0, max=10.0, step=0.5, description="clahe_clip", layout=_half)
w_cross_gain = FloatSlider(value=2.0, min=0.5, max=10.0, step=0.5, description="cross_gain", layout=_half)
w_diff_gain = FloatSlider(value=4.0, min=1.0, max=20.0, step=0.5, description="diff_gain", layout=_half)

# --- ROI geometry sliders (cross-track widening especially useful when ADS-B
# alignment is imperfect; along-track growth catches longer contrails). ---
w_roi_along = IntSlider(value=int(DET_CFG.roi_along_px), min=60, max=600, step=10, description="roi_along", layout=_half)
w_path_baseline = IntSlider(value=10, min=1, max=30, step=1, description="path_baseline_s", layout=_half)
w_roi_cross = IntSlider(value=int(DET_CFG.roi_cross_px), min=20, max=240, step=5, description="roi_cross", layout=_half)

# --- Detector knobs ---
w_use_adapt = Checkbox(value=bool(DET_CFG.use_adaptive_canny), description="adaptive Canny")
w_use_mask = Checkbox(value=bool(DET_CFG.use_rotated_mask), description="rotated mask")

w_pct_high = FloatSlider(value=float(DET_CFG.canny_percentile_high), min=95.0, max=99.9, step=0.1, description="pct_high", layout=_half)
w_pct_low = FloatSlider(value=float(DET_CFG.canny_percentile_low), min=80.0, max=99.0, step=0.5, description="pct_low", layout=_half)
w_low_ratio = FloatSlider(value=float(DET_CFG.canny_low_ratio), min=0.1, max=0.8, step=0.05, description="low_ratio", layout=_half)
w_min_high = IntSlider(value=int(DET_CFG.canny_min_high), min=10, max=200, step=5, description="min_high", layout=_half)

w_canny_low = IntSlider(value=int(DET_CFG.canny_low), min=1, max=255, step=1, description="canny_low", layout=_half)
w_canny_high = IntSlider(value=int(DET_CFG.canny_high), min=1, max=400, step=1, description="canny_high", layout=_half)

w_blur = IntSlider(value=int(DET_CFG.blur_kernel), min=0, max=11, step=1, description="blur", layout=_half)

w_hough_thr = IntSlider(value=int(DET_CFG.hough_threshold), min=5, max=100, step=1, description="hough_thr", layout=_half)
w_hough_minL = IntSlider(value=int(DET_CFG.hough_min_line_length), min=5, max=80, step=1, description="hough_minL", layout=_half)
w_hough_gap = IntSlider(value=int(DET_CFG.hough_max_line_gap), min=1, max=30, step=1, description="hough_gap", layout=_half)

w_tol = FloatSlider(value=float(DET_CFG.angle_tolerance_deg), min=1.0, max=45.0, step=0.5, description="angle_tol", layout=_half)
w_longL = FloatSlider(value=float(DET_CFG.long_line_min_px), min=5.0, max=80.0, step=1.0, description="long_min", layout=_half)
w_score_norm = FloatSlider(value=float(DET_CFG.score_length_norm_px), min=30.0, max=400.0, step=10.0, description="score_norm_px", layout=_half)

controls = VBox([
    w_cand,
    Label(value="Transform chain (applied left-to-right before Canny):"),
    HBox([w_ch1, w_ch2, w_ch3]),
    Label(value="Per-transform parameters (only affect chains using that transform):"),
    HBox([w_lc_sigma, w_dog_lo]),
    HBox([w_dog_hi, w_frangi_lo]),
    HBox([w_frangi_hi, w_clahe_clip]),
    HBox([w_cross_gain, w_diff_gain]),
    Label(value="ROI polygon geometry (widening cross-track compensates for ADS-B misalignment):"),
    HBox([w_roi_along, w_roi_cross]),
    HBox([w_path_baseline]),
    Label(value="Canny + Hough knobs:"),
    HBox([w_use_adapt, w_use_mask]),
    HBox([w_pct_high, w_pct_low]),
    HBox([w_low_ratio, w_min_high]),
    HBox([w_canny_low, w_canny_high]),
    HBox([w_blur, w_hough_thr]),
    HBox([w_hough_minL, w_hough_gap]),
    HBox([w_tol, w_longL]),
    HBox([w_score_norm]),
])

def _current_chain() -> list[str]:
    return [w_ch1.value, w_ch2.value, w_ch3.value]

def _current_transform_params() -> dict:
    return {
        "local_contrast": {"local_contrast_sigma": float(w_lc_sigma.value)},
        "dog": {"dog_sigma_low": float(w_dog_lo.value), "dog_sigma_high": float(w_dog_hi.value)},
        "frangi": {
            "frangi_sigma_min": float(w_frangi_lo.value),
            "frangi_sigma_max": float(w_frangi_hi.value),
        },
        "clahe": {"clahe_clip_limit": float(w_clahe_clip.value)},
        "cross_grad": {"cross_grad_gain": float(w_cross_gain.value)},
        "temporal_diff": {"temporal_diff_gain": float(w_diff_gain.value)},
    }

def _current_knobs() -> dict:
    return {
        "use_adaptive_canny": w_use_adapt.value,
        "use_rotated_mask": w_use_mask.value,
        "canny_percentile_high": w_pct_high.value,
        "canny_percentile_low": w_pct_low.value,
        "canny_low_ratio": w_low_ratio.value,
        "canny_min_high": w_min_high.value,
        "canny_low": w_canny_low.value,
        "canny_high": w_canny_high.value,
        "blur_kernel": w_blur.value,
        "hough_threshold": w_hough_thr.value,
        "hough_min_line_length": w_hough_minL.value,
        "hough_max_line_gap": w_hough_gap.value,
        "angle_tolerance_deg": w_tol.value,
        "long_line_min_px": w_longL.value,
        "score_length_norm_px": w_score_norm.value,
        "roi_along_px": w_roi_along.value,
        "roi_cross_px": w_roi_cross.value,
        "path_baseline_s": int(w_path_baseline.value),
    }

In [15]:
render_out = Output()


def _angle_delta(a: float, b: float) -> float:
    return abs(((a - b + 90.0) % 180.0) - 90.0)


def _render_panels(**kwargs):
    idx = kwargs["cand_idx"]
    rec = RECORDS[idx]
    chain = _current_chain()
    params = _current_transform_params()
    knobs = _current_knobs()

    gray, result, rect, poly, path_vec, cfg = run_chain_and_detect(rec, chain, params, knobs)

    # Replicate Canny+Hough intermediate artefacts for the edge panel.
    mask = None
    base = gray
    if cfg.blur_kernel and cfg.blur_kernel > 1:
        k = int(cfg.blur_kernel) | 1
        base = cv2.GaussianBlur(base, (k, k), 0)
    if cfg.use_rotated_mask:
        mask = np.zeros(base.shape, dtype=np.uint8)
        cv2.fillPoly(mask, [poly.astype(np.int32)], 255)
        masked_values = base[mask > 0]
    else:
        masked_values = base.reshape(-1)
    if cfg.use_adaptive_canny and masked_values.size:
        p_hi = float(np.percentile(masked_values, cfg.canny_percentile_high))
        p_lo = float(np.percentile(masked_values, cfg.canny_percentile_low))
        canny_high = max(int(round(p_hi)), int(cfg.canny_min_high))
        canny_low = max(1, int(round(canny_high * cfg.canny_low_ratio)))
        floor = int(round(p_lo))
    else:
        canny_high, canny_low, floor = int(cfg.canny_high), int(cfg.canny_low), 0
    crop_for_canny = base.copy()
    if mask is not None:
        crop_for_canny = cv2.bitwise_and(crop_for_canny, crop_for_canny, mask=mask)
    if floor > 0:
        _, crop_for_canny = cv2.threshold(crop_for_canny, floor, 255, cv2.THRESH_TOZERO)
    edges = cv2.Canny(crop_for_canny, canny_low, canny_high)
    if mask is not None:
        edges = cv2.bitwise_and(edges, edges, mask=mask)
    raw = cv2.HoughLinesP(
        edges, rho=1, theta=np.pi / 180.0,
        threshold=int(cfg.hough_threshold),
        minLineLength=int(cfg.hough_min_line_length),
        maxLineGap=int(cfg.hough_max_line_gap),
    )
    raw_lines = [] if raw is None else [tuple(int(v) for v in ln[0]) for ln in raw]

    path_angle = math.degrees(math.atan2(path_vec[1], path_vec[0])) % 180.0
    tol = float(cfg.angle_tolerance_deg)

    def _aligned(x1, y1, x2, y2):
        a = math.degrees(math.atan2(y2 - y1, x2 - x1)) % 180.0
        return _angle_delta(a, path_angle) <= tol

    with render_out:
        render_out.clear_output(wait=True)
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        crop_rgb = cv2.cvtColor(rec["crop"], cv2.COLOR_BGR2RGB)
        tl_x, tl_y = rec["crop_tl"]
        cx = rec["pixel_x"] - tl_x
        cy = rec["pixel_y"] - tl_y

        # Panel 1: crop with rotated polygon + backward-only path line.
        axes[0, 0].imshow(crop_rgb)
        poly_closed = np.vstack([poly, poly[:1]])
        axes[0, 0].plot(poly_closed[:, 0], poly_closed[:, 1], color="#ffb000", lw=1.5)
        # Backward-only: from aircraft toward where it came from.
        L = 60.0
        axes[0, 0].plot(
            [cx, cx - L * path_vec[0]],
            [cy, cy - L * path_vec[1]],
            color="#ff6030", lw=1.4, alpha=0.9,
        )
        axes[0, 0].scatter([cx], [cy], c="#ff6030", s=24, zorder=5)
        axes[0, 0].set_title(
            f"{rec['id']} {rec['callsign']}  [{_label_tag(rec)}, {rec['stratum']}]  "
            f"crop {crop_rgb.shape[1]}×{crop_rgb.shape[0]}  path={path_angle:5.1f}°"
        )
        axes[0, 0].axis("off")

        # Panel 2: chain output (what detect() sees).
        last_tf = next((c for c in reversed(chain) if c and c != "none"), None)
        cmap = DISPLAY_COLORMAPS.get(last_tf, "gray") if last_tf else "gray"
        axes[0, 1].imshow(gray, cmap=cmap)
        chain_str = " → ".join(c for c in chain if c and c != "none") or "(passthrough)"
        axes[0, 1].set_title(f"chain output: {chain_str}")
        axes[0, 1].axis("off")

        # Panel 3: Canny edges.
        axes[1, 0].imshow(edges, cmap="gray")
        axes[1, 0].set_title(
            f"Canny edges  canny_low={canny_low} canny_high={canny_high} floor={floor}"
        )
        axes[1, 0].axis("off")

        # Panel 4: Hough overlay on the crop.
        axes[1, 1].imshow(crop_rgb)
        axes[1, 1].plot(poly_closed[:, 0], poly_closed[:, 1], color="#ffb000", lw=1.0)
        n_aligned = n_rejected = 0
        for (x1, y1, x2, y2) in raw_lines:
            if _aligned(x1, y1, x2, y2):
                axes[1, 1].plot([x1, x2], [y1, y2], color="#50e050", lw=1.0, alpha=0.9)
                n_aligned += 1
            else:
                axes[1, 1].plot([x1, x2], [y1, y2], color="#ff9a40", lw=0.8, alpha=0.55)
                n_rejected += 1
        if result.pixel_line is not None:
            x1, y1, x2, y2 = result.pixel_line
            axes[1, 1].plot([x1, x2], [y1, y2], color="#30ff30", lw=2.8)
        axes[1, 1].set_title(
            f"score={result.score:.3f}  long={result.num_long_lines}  aligned={n_aligned}  "
            f"rej={n_rejected}  len_px={result.contrail_length_px:.0f}"
        )
        axes[1, 1].axis("off")
        plt.tight_layout()
        plt.show()


_all_widgets = [
    w_cand, w_ch1, w_ch2, w_ch3,
    w_lc_sigma, w_dog_lo, w_dog_hi, w_frangi_lo, w_frangi_hi, w_clahe_clip,
    w_cross_gain, w_diff_gain,
    w_roi_along, w_roi_cross, w_path_baseline,
    w_use_adapt, w_use_mask, w_pct_high, w_pct_low, w_low_ratio, w_min_high,
    w_canny_low, w_canny_high, w_blur, w_hough_thr, w_hough_minL, w_hough_gap,
    w_tol, w_longL, w_score_norm,
]
ui_out = interactive_output(_render_panels, {"cand_idx": w_cand})
for w in _all_widgets:
    if w is not w_cand:
        w.observe(lambda *_: _render_panels(cand_idx=w_cand.value), names="value")

display(controls, render_out)

Output()

## Add user-suggested candidates

The 35-candidate stratified batch can't be the whole picture — you've seen
contrails in the April-9 video on other flights that didn't land in the
batch. This cell lets you add them as labelled records in `RECORDS` so the
matrix / Bulk AUC / live-render pick them up.

Format of `USER_CANDIDATES` (edit the list below):
- `(callsign, time_iso_utc, label, note)`
- `time_iso_utc` is the UTC peak moment for the contrail (e.g.
  `"2026-04-09T13:14:00+00:00"`). If you have an ET time, convert it: ET on
  April 9 is UTC-4 (DST), so 9:50 ET = 13:50 UTC.
- `label` is one of `"positive"` (contrail) / `"negative"` (no contrail) /
  `"unsure"`.
- `note` is free-form (optional), surfaced in the dropdown and matrix title.

After editing, run the cell to re-decode the relevant video frames, append
records with source=`"user"` and stratum=`"user_added"`, and rebuild the
widget dropdown so the new rows appear in the candidate selector.

In [16]:
# Edit this list; re-run the cell to refresh RECORDS.
USER_CANDIDATES: list[tuple[str, str, str, str]] = [
    # (callsign, time_iso_utc, label, note)
    ("MNB312", "2026-04-09T13:14:19+00:00", "positive", "seen in video"),
    ("CHG591", "2026-04-09T13:51:16+00:00", "positive", "seen in video"),
    ("AFR2N",  "2026-04-09T13:54:47+00:00", "positive", "seen in video"),
    ("VIR91U", "2026-04-09T14:08:59+00:00", "positive", "seen in video"),
]


def _resolve_user_candidate(callsign: str, t_iso: str, label: str, note: str):
    t_target = datetime.datetime.fromisoformat(t_iso).astimezone(timezone.utc)
    best = None
    best_dt = None
    for flight in ADSB:
        if (flight.get("callsign") or "").strip().upper() != callsign.upper():
            continue
        for i, p in enumerate(flight["pings"]):
            pt = datetime.datetime.fromisoformat(p["time"]).astimezone(timezone.utc)
            dt = abs((pt - t_target).total_seconds())
            if best_dt is None or dt < best_dt:
                best_dt = dt
                best = (flight, i, p, pt)
    if best is None:
        print(f"  WARN: {callsign} not found in adsb.json; skipping")
        return None
    flight, i_ping, ping, ping_t = best
    if best_dt > 60.0:
        print(f"  WARN: {callsign} nearest ping is {best_dt:.0f}s from target; skipping")
        return None

    proj = project_pings([_ping_from_dict(ping)], CALIB)[0]
    if proj is None:
        print(f"  WARN: {callsign} at {ping_t.strftime('%H:%M:%S')}Z projects outside frame; skipping")
        return None

    # Initial path_dx/dy uses the same 1-second baseline as the production
    # pipeline so the user can A/B it against longer baselines via the slider.
    pings_list = flight["pings"]
    i_back = max(0, i_ping - 1)
    i_fwd = min(len(pings_list) - 1, i_ping + 1)
    nbrs = project_pings(
        [_ping_from_dict(pings_list[i_back]), _ping_from_dict(pings_list[i_fwd])],
        CALIB,
    )
    if nbrs[0] is None or nbrs[1] is None:
        path_dx, path_dy = 1.0, 0.0
    else:
        dx = nbrs[1].x - nbrs[0].x
        dy = nbrs[1].y - nbrs[0].y
        mag = (dx * dx + dy * dy) ** 0.5
        path_dx, path_dy = (dx / mag, dy / mag) if mag > 1e-3 else (1.0, 0.0)

    frame_idx = int(round((ping_t - VIDEO_T0).total_seconds()))
    if frame_idx < 0:
        print(f"  WARN: {callsign} frame_idx={frame_idx} is negative; skipping")
        return None

    return {
        "callsign": callsign,
        "transponder_id": flight["transponder_id"],
        "pings": pings_list,
        "peak_ping_idx": i_ping,
        "pixel_x": proj.x,
        "pixel_y": proj.y,
        "path_dx": path_dx,
        "path_dy": path_dy,
        "frame_idx": frame_idx,
        "ping_time": ping_t,
        "label": label,
        "note": note,
    }


user_resolved = [_resolve_user_candidate(*c) for c in USER_CANDIDATES]
user_resolved = [r for r in user_resolved if r is not None]
user_needed_frames: set[int] = set()
for r in user_resolved:
    user_needed_frames.add(r["frame_idx"])
    if r["frame_idx"] > 0:
        user_needed_frames.add(r["frame_idx"] - 1)

already = {k for k in frames.keys()}
missing = user_needed_frames - already
if missing:
    print(f"Decoding {len(missing)} additional frames for user candidates ...")
    new_frames = _decode_frames(APRIL9_VIDEO_LINK, missing)
    frames.update(new_frames)

RECORDS[:] = [r for r in RECORDS if r["source"] != "user"]

for r in user_resolved:
    fidx = r["frame_idx"]
    frame = frames.get(fidx)
    if frame is None:
        print(f"  SKIP {r['callsign']}: frame {fidx} missing")
        continue
    crop, (tl_x, tl_y) = _context_crop(frame, r["pixel_x"], r["pixel_y"], CROP_PAD_PX)
    prev_crop = None
    if fidx > 0 and (fidx - 1) in frames:
        prev_crop, _ = _context_crop(frames[fidx - 1], r["pixel_x"], r["pixel_y"], CROP_PAD_PX)
    RECORDS.append({
        "source": "user",
        "id": f"U#{r['callsign']}",
        "label": r["label"],
        "stratum": "user_added",
        "callsign": r["callsign"],
        "transponder_id": r["transponder_id"],
        "frame_idx": fidx,
        "pixel_x": r["pixel_x"],
        "pixel_y": r["pixel_y"],
        "path_dx": r["path_dx"],
        "path_dy": r["path_dy"],
        "roi": {"x": int(r["pixel_x"]) - 90, "y": int(r["pixel_y"]) - 20, "w": 180, "h": 40},
        "crop": crop,
        "prev_crop": prev_crop,
        "crop_tl": (tl_x, tl_y),
        "pings": r["pings"],
        "peak_ping_idx": r["peak_ping_idx"],
        "note": r["note"],
    })
    print(
        f"  + {r['callsign']} @ {r['ping_time'].strftime('%H:%M:%S')}Z  "
        f"pixel=({r['pixel_x']:.0f}, {r['pixel_y']:.0f})  label={r['label']}"
    )

w_cand.options = [
    (f"{r['id']}  {r['callsign']:<10}  [{_label_tag(r)}, {r['stratum']}]", i)
    for i, r in enumerate(RECORDS)
]

a9_user_pos = sum(1 for r in RECORDS if r["source"] == "user" and r["label"] == "positive")
a9_user_neg = sum(1 for r in RECORDS if r["source"] == "user" and r["label"] == "negative")
print(
    f"\nUser candidates merged. Total RECORDS: {len(RECORDS)} "
    f"(user positive={a9_user_pos}, user negative={a9_user_neg})"
)

April-9 video starts at frame 0 = 2026-04-09T04:00:01+00:00
Decoding 8 additional frames for user candidates ...
  + MNB312 @ 13:14:19Z  pixel=(3093, 653)  label=positive
  + CHG591 @ 13:51:16Z  pixel=(3236, 643)  label=positive
  + AFR2N @ 13:54:47Z  pixel=(3235, 611)  label=positive
  + VIR91U @ 14:08:59Z  pixel=(2942, 439)  label=positive

User candidates merged. Total RECORDS: 46 (user positive=4, user negative=0)


## Matrix of positives

Render the 4-panel view for **every labelled positive** (April-9 contrails
+ April-8 sentinels) at the current chain + knob settings, in one grid.
Each tile is colour-coded: green border = TP at current Youden-J
threshold, red border = FN.

Great for regression-spotting: if a tweak that boosts April-9 recall also
pushes an April-8 sentinel below threshold, the red border makes it obvious.

In [18]:
positives_grid_out = Output()


def render_positives_matrix(_button=None):
    chain = _current_chain()
    params = _current_transform_params()
    knobs = _current_knobs()

    positives = [r for r in RECORDS if r["label"] == "positive"]
    if not positives:
        with positives_grid_out:
            positives_grid_out.clear_output()
            print("No positives in RECORDS.")
        return

    # Score everything once so we can derive the Youden-J threshold.
    results_by_id: dict[str, dict] = {}
    for rec in RECORDS:
        gray, result, rect, poly, pv, cfg = run_chain_and_detect(rec, chain, params, knobs)
        results_by_id[rec["id"]] = {
            "rec": rec, "gray": gray, "result": result,
            "poly": poly, "path_vec": pv, "cfg": cfg,
        }

    a9_pos = [
        results_by_id[r["id"]]["result"].score for r in RECORDS
        if r["source"] == "april-9" and r["label"] == "positive"
    ]
    a9_neg = [
        results_by_id[r["id"]]["result"].score for r in RECORDS
        if r["source"] == "april-9" and r["label"] == "negative"
    ]
    if a9_pos and a9_neg:
        scores = sorted(set(a9_pos + a9_neg))
        cands = [(a + b) / 2 for a, b in zip(scores[:-1], scores[1:])] or [0.0]
        best_t, best_j = cands[0], -1.0
        for t in cands:
            tpr = sum(1 for p in a9_pos if p >= t) / len(a9_pos)
            fpr = sum(1 for n in a9_neg if n >= t) / len(a9_neg)
            if tpr - fpr > best_j:
                best_j, best_t = tpr - fpr, t
        threshold = float(best_t)
    else:
        threshold = 0.083

    n = len(positives)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.2, nrows * 3.8))
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes.reshape(1, -1)
    elif ncols == 1:
        axes = axes.reshape(-1, 1)

    tp_count = 0
    for idx, rec in enumerate(positives):
        r, c = divmod(idx, ncols)
        ax = axes[r, c]
        info = results_by_id[rec["id"]]
        score = float(info["result"].score)
        is_tp = score >= threshold
        if is_tp:
            tp_count += 1
        border_color = "#30c030" if is_tp else "#e03030"

        crop_rgb = cv2.cvtColor(rec["crop"], cv2.COLOR_BGR2RGB)
        ax.imshow(crop_rgb)

        poly = info["poly"]
        poly_closed = np.vstack([poly, poly[:1]])
        ax.plot(poly_closed[:, 0], poly_closed[:, 1], color="#ffb000", lw=1.2)

        tl_x, tl_y = rec["crop_tl"]
        cx = rec["pixel_x"] - tl_x
        cy = rec["pixel_y"] - tl_y
        L = 60.0
        ax.plot(
            [cx, cx - L * info["path_vec"][0]],
            [cy, cy - L * info["path_vec"][1]],
            color="#ff6030", lw=1.2, alpha=0.9,
        )
        ax.scatter([cx], [cy], c="#ff6030", s=18, zorder=5)

        if info["result"].pixel_line is not None:
            x1, y1, x2, y2 = info["result"].pixel_line
            ax.plot([x1, x2], [y1, y2], color="#30ff30", lw=2.2)

        tag = "TP" if is_tp else "FN"
        ax.set_title(
            f"{rec['id']}  {rec['callsign']}  [{rec['stratum']}]\n"
            f"{tag}   score={score:.3f}",
            color=border_color, fontsize=10,
        )
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(border_color)
            spine.set_linewidth(3)

    for idx in range(len(positives), nrows * ncols):
        r, c = divmod(idx, ncols)
        axes[r, c].axis("off")

    chain_str = " → ".join(c for c in chain if c and c != "none") or "(passthrough)"
    fig.suptitle(
        f"Positives matrix - chain: {chain_str}   threshold={threshold:.3f}   "
        f"TP={tp_count}/{len(positives)}",
        y=1.0, fontsize=12,
    )

    with positives_grid_out:
        positives_grid_out.clear_output(wait=True)
        plt.tight_layout()
        plt.show()


matrix_button = Button(
    description="Render positives matrix",
    button_style="info",
    layout=Layout(width="280px"),
)
matrix_button.on_click(render_positives_matrix)
display(matrix_button, positives_grid_out)

Button(button_style='info', description='Render positives matrix', layout=Layout(width='280px'), style=ButtonS…

Output()

## Bulk evaluation

Press **Run Bulk AUC** to re-run the current chain + knobs over all 34 April-9
labels plus the 7 April-8 positive sentinels. Results show:

- **Overall AUC** (Mann-Whitney estimate), Youden-J threshold
- **Recall @ Youden-J** and **FP count @ Youden-J** on the April-9 set
- **Per-stratum recall** (high_cirrus / mid_cruise / wide_radius / marginal)
- **April-8 sentinel pass rate** at the same threshold
- **GO / NO-GO** verdict against the PRD item 29 criterion
- Histogram of positive vs negative scores with the threshold line overlaid

Evaluation takes ≈5 s for a 34-candidate run.

In [19]:
bulk_out = Output()


def _mann_whitney_auc(pos: list[float], neg: list[float]) -> float:
    if not pos or not neg:
        return 0.5
    wins = 0.0
    for p in pos:
        for n in neg:
            if p > n:
                wins += 1.0
            elif p == n:
                wins += 0.5
    return wins / (len(pos) * len(neg))


def _best_threshold(pos: list[float], neg: list[float]) -> tuple[float, float]:
    if not pos or not neg:
        return 0.5, 0.0
    scores = sorted(set(pos + neg))
    if len(scores) == 1:
        return scores[0] - 1e-6, 0.0
    cands = [(a + b) / 2 for a, b in zip(scores[:-1], scores[1:])]
    best_t, best_j = cands[0], -1.0
    for t in cands:
        tpr = sum(1 for p in pos if p >= t) / len(pos)
        fpr = sum(1 for n in neg if n >= t) / len(neg)
        if tpr - fpr > best_j:
            best_j, best_t = tpr - fpr, t
    return best_t, best_j


def _score_records(records, chain, params, knobs) -> list[float]:
    scores = []
    for rec in records:
        _, result, *_ = run_chain_and_detect(rec, chain, params, knobs)
        scores.append(float(result.score))
    return scores


def run_bulk_eval(_button=None):
    chain = _current_chain()
    params = _current_transform_params()
    knobs = _current_knobs()

    # Score everything in one pass.
    all_scores: dict[str, float] = {}
    for rec in RECORDS:
        _, result, *_ = run_chain_and_detect(rec, chain, params, knobs)
        all_scores[rec["id"]] = float(result.score)

    # April-9 split
    a9_pos, a9_neg, a9_unsure = [], [], []
    stratum_scores: dict[str, list[tuple[float, str]]] = defaultdict(list)
    for rec in RECORDS:
        if rec["source"] != "april-9":
            continue
        s = all_scores[rec["id"]]
        stratum_scores[rec["stratum"]].append((s, rec["label"] or "unlabeled"))
        if rec["label"] == "positive":
            a9_pos.append(s)
        elif rec["label"] == "negative":
            a9_neg.append(s)
        elif rec["label"] == "unsure":
            a9_unsure.append(s)

    # April-8 sentinels
    a8_pos = [all_scores[r["id"]] for r in RECORDS if r["source"] == "april-8"]

    auc = _mann_whitney_auc(a9_pos, a9_neg)
    threshold, youden_j = _best_threshold(a9_pos, a9_neg)

    recall = sum(1 for p in a9_pos if p >= threshold)
    fp_count = sum(1 for n in a9_neg if n >= threshold)
    a8_pass = sum(1 for p in a8_pos if p >= threshold)

    # Per-stratum recall (positives only)
    stratum_lines = []
    strata_with_recall = 0
    for stratum in ("high_cirrus", "mid_cruise", "wide_radius", "marginal"):
        pos_items = [s for s, lab in stratum_scores.get(stratum, []) if lab == "positive"]
        tp = sum(1 for s in pos_items if s >= threshold)
        if pos_items:
            stratum_lines.append(f"  {stratum:<12} recall {tp}/{len(pos_items)}")
            if tp > 0:
                strata_with_recall += 1
        else:
            stratum_lines.append(f"  {stratum:<12} (no positives)")

    go_recall = recall >= 10 and len(a9_pos) >= 15
    go_fp = fp_count <= 2
    go_strata = strata_with_recall >= 3
    verdict = "GO" if (go_recall and go_fp and go_strata) else "NO-GO"

    chain_str = " → ".join(c for c in chain if c and c != "none") or "(passthrough)"

    with bulk_out:
        bulk_out.clear_output(wait=True)
        print(f"=== Bulk evaluation — chain: {chain_str} ===")
        print()
        print(f"April-9 labeled: {len(a9_pos)} positives, {len(a9_neg)} negatives, {len(a9_unsure)} unsure")
        print(f"April-8 sentinels: {len(a8_pos)} positives")
        print()
        print(f"AUC (Mann-Whitney):      {auc:.3f}")
        print(f"Youden-J threshold:      {threshold:.3f}  (J = {youden_j:.3f})")
        print(f"April-9 recall @ t:      {recall}/{len(a9_pos)}   "
              f"({'PASS' if go_recall else 'FAIL'} ― need ≥10/15)")
        print(f"April-9 FP @ t:          {fp_count}/{len(a9_neg)}   "
              f"({'PASS' if go_fp else 'FAIL'} ― need ≤2/19)")
        print(f"April-8 sentinel @ t:    {a8_pass}/{len(a8_pos)}   (regression check)")
        print()
        print("Per-stratum recall:")
        for line in stratum_lines:
            print(line)
        print(f"  Strata with any recall:  {strata_with_recall}/4   "
              f"({'PASS' if go_strata else 'FAIL'} ― need ≥3/4)")
        print()
        print(f"VERDICT: {verdict}")

        fig, ax = plt.subplots(figsize=(10, 3.5))
        edges = np.linspace(0, 1, 21)
        if a9_neg:
            ax.hist(a9_neg, bins=edges, alpha=0.6, label=f"A9 negative (n={len(a9_neg)})", color="#3c78d8")
        if a9_pos:
            ax.hist(a9_pos, bins=edges, alpha=0.6, label=f"A9 positive (n={len(a9_pos)})", color="#e06666")
        if a8_pos:
            ax.hist(a8_pos, bins=edges, alpha=0.5, label=f"A8 sentinel (n={len(a8_pos)})", color="#f1c232")
        ax.axvline(threshold, color="k", ls="--", label=f"Youden-J thr={threshold:.3f}")
        ax.set_title(
            f"AUC={auc:.3f}  recall={recall}/{len(a9_pos)}  FP={fp_count}/{len(a9_neg)}  "
            f"strata={strata_with_recall}/4  — {verdict}"
        )
        ax.set_xlabel("detector score")
        ax.set_ylabel("count")
        ax.legend(loc="upper right", fontsize=9)
        plt.tight_layout()
        plt.show()


run_button = Button(description="Run Bulk AUC", button_style="primary", layout=Layout(width="200px"))
run_button.on_click(run_bulk_eval)
display(run_button, bulk_out)

Button(button_style='primary', description='Run Bulk AUC', layout=Layout(width='200px'), style=ButtonStyle())

Output()

## Export current settings

Once you've found a chain + knob configuration you like, run the cell below
to print a YAML block you can paste into `configs/mit_green_building.yaml`.

Note: wiring `transform_chain` into `concam.detection.detect()` so the live
pipeline honours it is an explicit follow-up PRD item — this notebook
produces the YAML snippet but you'll need that follow-up item to actually
apply it to full-day runs.

In [8]:
def export_yaml():
    chain = [c for c in _current_chain() if c and c != "none"]
    params = _current_transform_params()
    knobs = _current_knobs()

    def _fmt(v):
        if isinstance(v, bool):
            return str(v).lower()
        if isinstance(v, float):
            return f"{v:g}"
        return str(v)

    out = ["detection:"]
    out.append(f"  # Chain applied left-to-right before Canny:")
    out.append(f"  transform_chain: [{', '.join(chain) if chain else '# empty — passthrough'}]")
    # Include only parameters for transforms currently in the chain.
    active_params = {k: v for k, v in params.items() if k in chain}
    if active_params:
        out.append("  transform_params:")
        for tf_name, pkvs in active_params.items():
            out.append(f"    {tf_name}:")
            for pk, pv in pkvs.items():
                out.append(f"      {pk}: {_fmt(pv)}")
    out.append(f"  use_adaptive_canny: {_fmt(knobs['use_adaptive_canny'])}")
    out.append(f"  use_rotated_mask: {_fmt(knobs['use_rotated_mask'])}")
    out.append(f"  canny_percentile_high: {_fmt(knobs['canny_percentile_high'])}")
    out.append(f"  canny_percentile_low: {_fmt(knobs['canny_percentile_low'])}")
    out.append(f"  canny_low_ratio: {_fmt(knobs['canny_low_ratio'])}")
    out.append(f"  canny_min_high: {_fmt(knobs['canny_min_high'])}")
    out.append(f"  canny_low: {_fmt(knobs['canny_low'])}")
    out.append(f"  canny_high: {_fmt(knobs['canny_high'])}")
    out.append(f"  blur_kernel: {_fmt(knobs['blur_kernel'])}")
    out.append(f"  hough_threshold: {_fmt(knobs['hough_threshold'])}")
    out.append(f"  hough_min_line_length: {_fmt(knobs['hough_min_line_length'])}")
    out.append(f"  hough_max_line_gap: {_fmt(knobs['hough_max_line_gap'])}")
    out.append(f"  angle_tolerance_deg: {_fmt(knobs['angle_tolerance_deg'])}")
    out.append(f"  long_line_min_px: {_fmt(knobs['long_line_min_px'])}")
    out.append(f"  score_length_norm_px: {_fmt(knobs['score_length_norm_px'])}")
    out.append(f"  roi_along_px: {ROI_ALONG_PX}")
    out.append(f"  roi_cross_px: {ROI_CROSS_PX}")
    print("\n".join(out))


export_yaml()

detection:
  # Chain applied left-to-right before Canny:
  transform_chain: [cross_grad]
  transform_params:
    cross_grad:
      cross_grad_gain: 0.5
  use_adaptive_canny: true
  use_rotated_mask: false
  canny_percentile_high: 99.7
  canny_percentile_low: 97
  canny_low_ratio: 0.25
  canny_min_high: 60
  canny_low: 50
  canny_high: 150
  blur_kernel: 3
  hough_threshold: 15
  hough_min_line_length: 12
  hough_max_line_gap: 10
  angle_tolerance_deg: 12
  long_line_min_px: 25
  score_length_norm_px: 80
  roi_along_px: 180
  roi_cross_px: 40
